# PAC-3310 Functional-Profile Figure

Functional characterization of PAC-3310 across muscarinic receptor subtypes. This notebook reads the final replicate-level outputs from the consolidated calcium-flux and cAMP analyses and creates the publication-style four-panel functional-profile figure (Figure 1 in the current report):

- **A:** carbachol cAMP concentration-response curves at M2 and M4
- **B:** carbachol calcium-flux concentration-response curves at M1, M3, and M5
- **C:** PAC-3310 agonism across M1-M5
- **D:** PAC-3310 antagonism across M1-M5

Points are means and error bars are SEM. A 4PL curve is shown only when it is supported over a flat response at $p<0.01$, its midpoint lies inside the tested range, and its direction matches the expected pharmacology. Statistical testing uses the unconstrained 4PL. For display only, rendered Hill slopes are capped at 2.0 to prevent visually implausible near-vertical transitions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
from scipy.stats import f as f_dist

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
CA_REPLICATES = ROOT / 'data' / 'ca_assay' / 'processed' / 'PAC-3310_calcium_flux_CRC_replicates.csv'
CAMP_REPLICATES = ROOT / 'data' / 'camp_assay' / 'processed' / 'PAC-3310_cAMP_replicate_results.csv'
FIGURE_DIR = ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PNG = FIGURE_DIR / 'functional-profile.png'
OUTPUT_PDF = FIGURE_DIR / 'functional-profile.pdf'
OUTPUT_SVG = FIGURE_DIR / 'functional-profile.svg'

RECEPTORS = ['M1', 'M2', 'M3', 'M4', 'M5']
COLORS = {
    'M1': '#0072B2',
    'M2': '#D55E00',
    'M3': '#009E73',
    'M4': '#CC79A7',
    'M5': '#E69F00',
}
MARKERS = {'M1': 'o', 'M2': 's', 'M3': 'D', 'M4': '^', 'M5': 'v'}
DISPLAY_HILL_SLOPE_CAP = 2.0

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'axes.linewidth': 1.0,
    'xtick.labelsize': 11.5,
    'ytick.labelsize': 11.5,
    'legend.fontsize': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

## Load and harmonize the final analysis outputs

The cAMP reference curves are fitted using the unnormalized, dilution-corrected cAMP concentrations. Panel A then assigns 100% to each receptor's fitted low-concentration cAMP plateau, avoiding dependence on a noisy per-plate vehicle mean while retaining the fitted response magnitude.

The two combined PAC-3310 panels use assay-independent biological endpoints:

- **Antagonism:** each receptor series is baseline-centered to the mean response at the lowest tested PAC-3310 concentration ($-11$ log M). This is an intercept-only fitted baseline, so every series begins at 0% while retaining all concentration-dependent changes and their direction.
- **Agonism:** PAC-3310 response amplitude is expressed relative to the matching receptor's fitted maximal carbachol response. The PAC-3310 curve is never normalized to its own maximum.

In [ ]:
ca = pd.read_csv(CA_REPLICATES)
camp = pd.read_csv(CAMP_REPLICATES)

required_ca = {
    'Receptor', 'Assay Mode', 'Concentration (log M)', 'ΔF/F0',
    'Control-referenced % Inhibition',
}
required_camp = {
    'Receptor', 'Assay Mode', 'Concentration (log M)', 'Is Control',
    'Normalized Response', 'Real-sample cAMP (pmol/mL)',
}
assert required_ca.issubset(ca.columns), required_ca - set(ca.columns)
assert required_camp.issubset(camp.columns), required_camp - set(camp.columns)

ca['Concentration (log M)'] = pd.to_numeric(ca['Concentration (log M)'], errors='coerce')
camp['Concentration (log M)'] = pd.to_numeric(camp['Concentration (log M)'], errors='coerce')
camp_doses = camp.loc[~camp['Is Control'].astype(bool)].copy()

def reference_4pl(log_concentration, response_at_low, hill_slope, log_ec50, response_at_high):
    exponent = np.clip((log_ec50 - log_concentration) * hill_slope, -300, 300)
    return response_at_low + (response_at_high - response_at_low) / (
        1.0 + 10.0 ** exponent
    )


def fit_free_reference_curve(frame, response_column):
    x = frame['Concentration (log M)'].to_numpy(float)
    y = frame[response_column].to_numpy(float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    low_guess = np.mean(y[x == x.min()])
    high_guess = np.mean(y[x == x.max()])
    parameters, _ = curve_fit(
        reference_4pl,
        x,
        y,
        p0=[low_guess, 1.0, np.median(x), high_guess],
        bounds=(
            [-np.inf, 0.01, x.min() - 2.0, -np.inf],
            [np.inf, 50.0, x.max() + 2.0, np.inf],
        ),
        maxfev=30000,
    )
    return {
        'response_at_low': parameters[0],
        'hill_slope': parameters[1],
        'log_ec50': parameters[2],
        'response_at_high': parameters[3],
    }

panels = []

def add_panel(frame, panel, response_column):
    part = frame[['Receptor', 'Concentration (log M)', response_column]].copy()
    part = part.rename(columns={response_column: 'Response'}).dropna()
    part['Panel'] = panel
    panels.append(part)

reference_rows = []
camp_carbachol_fits = {}
camp_carbachol = camp_doses.loc[camp_doses['Assay Mode'].eq('Carbachol agonism')].copy()

# A: fit absolute cAMP first, then set the fitted low-concentration
# plateau to 100%. Individual observations remain free to scatter above
# or below that fitted baseline.
for receptor in ['M2', 'M4']:
    mask = camp_carbachol['Receptor'].eq(receptor)
    fit = fit_free_reference_curve(
        camp_carbachol.loc[mask], 'Real-sample cAMP (pmol/mL)'
    )
    fitted_baseline = fit['response_at_low']
    fitted_stimulated_plateau = fit['response_at_high']
    if fitted_baseline <= 0 or fitted_baseline <= fitted_stimulated_plateau:
        raise ValueError(f'Invalid fitted carbachol cAMP reference for {receptor}')
    camp_carbachol_fits[receptor] = fit
    camp_carbachol.loc[mask, 'cAMP (% Fitted Baseline)'] = (
        100.0
        * camp_carbachol.loc[mask, 'Real-sample cAMP (pmol/mL)']
        / fitted_baseline
    )
    reference_rows.append({
        'Receptor': receptor,
        'Assay': 'cAMP',
        'Fitted baseline': fitted_baseline,
        'Fitted maximal carbachol response': fitted_stimulated_plateau,
    })

add_panel(camp_carbachol, 'A', 'cAMP (% Fitted Baseline)')

# B: native calcium-flux reference-agonist endpoint.
add_panel(
    ca.loc[ca['Assay Mode'].eq('Carbachol agonism')],
    'B', 'ΔF/F0',
)

# D: begin every antagonism series at 0 by subtracting an intercept-only
# baseline fitted to the replicate responses at the lowest PAC-3310 dose.
# This shifts the vertical origin without changing SEM, curve direction,
# concentration dependence, or model-comparison statistics.
def baseline_center_antagonism(frame, response_column):
    centered_parts = []
    for receptor, receptor_frame in frame.groupby('Receptor', sort=False):
        receptor_frame = receptor_frame.copy()
        lowest_dose = receptor_frame['Concentration (log M)'].min()
        fitted_baseline = receptor_frame.loc[
            receptor_frame['Concentration (log M)'].eq(lowest_dose), response_column
        ].mean()
        receptor_frame['Baseline-centered inhibition (%)'] = (
            receptor_frame[response_column] - fitted_baseline
        )
        centered_parts.append(receptor_frame)
    return pd.concat(centered_parts, ignore_index=True)


ca_antagonism = baseline_center_antagonism(
    ca.loc[ca['Assay Mode'].eq('PAC-3310 antagonism')],
    'Control-referenced % Inhibition',
)
camp_antagonism = baseline_center_antagonism(
    camp_doses.loc[camp_doses['Assay Mode'].eq('PAC-3310 antagonism')],
    'Normalized Response',
)
add_panel(ca_antagonism, 'D', 'Baseline-centered inhibition (%)')
add_panel(camp_antagonism, 'D', 'Baseline-centered inhibition (%)')

# C, calcium flux: scale PAC-3310 ΔF/F0 to the fitted high-concentration
# plateau of the matching carbachol curve.
ca_pac_agonism = ca.loc[ca['Assay Mode'].eq('PAC-3310 agonism')].copy()
for receptor in ['M1', 'M3', 'M5']:
    carbachol = ca.loc[
        ca['Receptor'].eq(receptor)
        & ca['Assay Mode'].eq('Carbachol agonism')
    ]
    fit = fit_free_reference_curve(carbachol, 'ΔF/F0')
    fitted_maximum = fit['response_at_high']
    if fitted_maximum <= 0:
        raise ValueError(f'Invalid fitted carbachol calcium reference for {receptor}')
    mask = ca_pac_agonism['Receptor'].eq(receptor)
    ca_pac_agonism.loc[mask, 'Activation (% Fitted Max Carbachol)'] = (
        100.0 * ca_pac_agonism.loc[mask, 'ΔF/F0'] / fitted_maximum
    )
    reference_rows.append({
        'Receptor': receptor,
        'Assay': 'Calcium flux',
        'Fitted baseline': fit['response_at_low'],
        'Fitted maximal carbachol response': fitted_maximum,
    })

add_panel(ca_pac_agonism, 'C', 'Activation (% Fitted Max Carbachol)')

# C, cAMP: use a fitted low-concentration baseline for the PAC-3310 plate,
# then express its fractional cAMP decrease relative to the fractional
# response range of the matching fitted carbachol curve. This removes the
# noisy vehicle anchor without defining efficacy from PAC-3310's own maximum.
camp_pac_agonism = camp_doses.loc[camp_doses['Assay Mode'].eq('PAC-3310 agonism')].copy()
for receptor in ['M2', 'M4']:
    mask = camp_pac_agonism['Receptor'].eq(receptor)
    pac_fit = fit_free_reference_curve(
        camp_pac_agonism.loc[mask], 'Real-sample cAMP (pmol/mL)'
    )
    pac_fitted_baseline = pac_fit['response_at_low']
    carbachol_fit = camp_carbachol_fits[receptor]
    carbachol_fractional_maximum = (
        (carbachol_fit['response_at_low'] - carbachol_fit['response_at_high'])
        / carbachol_fit['response_at_low']
    )
    if pac_fitted_baseline <= 0 or carbachol_fractional_maximum <= 0:
        raise ValueError(f'Invalid fitted cAMP agonism scale for {receptor}')
    pac_fractional_response = (
        pac_fitted_baseline
        - camp_pac_agonism.loc[mask, 'Real-sample cAMP (pmol/mL)']
    ) / pac_fitted_baseline
    camp_pac_agonism.loc[mask, 'Activation (% Fitted Max Carbachol)'] = (
        100.0
        * pac_fractional_response
        / carbachol_fractional_maximum
    )

add_panel(camp_pac_agonism, 'C', 'Activation (% Fitted Max Carbachol)')

plot_df = pd.concat(panels, ignore_index=True)
plot_df['Receptor'] = pd.Categorical(plot_df['Receptor'], RECEPTORS, ordered=True)
plot_df = plot_df.sort_values(['Panel', 'Receptor', 'Concentration (log M)']).reset_index(drop=True)

expected_panel_receptors = {
    'A': {'M2', 'M4'},
    'B': {'M1', 'M3', 'M5'},
    'C': set(RECEPTORS),
    'D': set(RECEPTORS),
}
observed_panel_receptors = {
    panel: set(group['Receptor'].astype(str))
    for panel, group in plot_df.groupby('Panel', observed=True)
}
assert observed_panel_receptors == expected_panel_receptors, observed_panel_receptors

reference_scales = pd.DataFrame(reference_rows)

In [ ]:
def model_4pl(log_concentration, response_at_low, hill_slope, log_ec50, response_at_high):
    exponent = np.clip((log_ec50 - log_concentration) * hill_slope, -300, 300)
    return response_at_low + (response_at_high - response_at_low) / (
        1.0 + 10.0 ** exponent
    )


def fit_4pl_vs_flat(frame, expected_direction, alpha=0.01):
    x = frame['Concentration (log M)'].to_numpy(float)
    y = frame['Response'].to_numpy(float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(y) <= 4 or np.unique(x).size < 4:
        return {'interpretable': False}

    low_guess = np.mean(y[x == x.min()])
    high_guess = np.mean(y[x == x.max()])
    try:
        popt, _ = curve_fit(
            model_4pl,
            x,
            y,
            p0=[low_guess, 1.0, np.median(x), high_guess],
            bounds=(
                [-np.inf, 0.01, x.min() - 2.0, -np.inf],
                [np.inf, 50.0, x.max() + 2.0, np.inf],
            ),
            maxfev=30000,
        )
    except Exception:
        return {'interpretable': False}

    fitted = model_4pl(x, *popt)
    ss_model = np.sum((y - fitted) ** 2)
    ss_flat = np.sum((y - y.mean()) ** 2)
    df_model = len(y) - 4
    if df_model <= 0 or ss_model <= 0 or ss_flat <= ss_model:
        f_statistic, p_value = np.nan, 1.0
    else:
        f_statistic = ((ss_flat - ss_model) / 3.0) / (ss_model / df_model)
        p_value = 1.0 - f_dist.cdf(f_statistic, 3, df_model)

    direction = 'increasing' if popt[3] > popt[0] else 'decreasing'
    midpoint_in_range = x.min() <= popt[2] <= x.max()
    interpretable = p_value < alpha and midpoint_in_range and direction == expected_direction
    return {
        'interpretable': interpretable,
        'parameters': popt,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'direction': direction,
        'midpoint_in_range': midpoint_in_range,
    }


expected_direction = {'A': 'decreasing', 'B': 'increasing', 'C': 'increasing', 'D': 'increasing'}
fits = {
    (panel, str(receptor)): fit_4pl_vs_flat(group, expected_direction[panel])
    for (panel, receptor), group in plot_df.groupby(['Panel', 'Receptor'], observed=True)
}

In [ ]:
panel_specs = {
    'A': {
        'title': 'Carbachol cAMP response',
        'xlabel': 'Carbachol concentration (log M)',
        'ylabel': 'cAMP (% fitted baseline)',
        'receptors': ['M2', 'M4'],
    },
    'B': {
        'title': 'Carbachol calcium-flux response',
        'xlabel': 'Carbachol concentration (log M)',
        'ylabel': r'$\Delta F/F_0$',
        'receptors': ['M1', 'M3', 'M5'],
    },
    'C': {
        'title': 'PAC-3310 agonism',
        'xlabel': 'PAC-3310 concentration (log M)',
        'ylabel': 'Activation (% fitted max carbachol response)',
        'receptors': RECEPTORS,
    },
    'D': {
        'title': 'PAC-3310 antagonism',
        'xlabel': 'PAC-3310 concentration (log M)',
        'ylabel': 'Baseline-centered signaling inhibition (%)',
        'receptors': RECEPTORS,
    },
}


def rounded_lower_bound(panel, step=20):
    grouped = (
        plot_df.loc[plot_df['Panel'].eq(panel)]
        .groupby(['Receptor', 'Concentration (log M)'], observed=True)['Response']
        .agg(['mean', 'sem'])
    )
    observed_min = np.nanmin((grouped['mean'] - grouped['sem'].fillna(0)).to_numpy())
    return min(-5.0, step * np.floor((observed_min - 3.0) / step))


def format_ec50(log_ec50):
    ec50_molar = 10.0 ** log_ec50
    if ec50_molar >= 1e-6:
        return f'{ec50_molar * 1e6:.2g} µM'
    return f'{ec50_molar * 1e9:.2g} nM'


def draw_panel(ax, panel):
    spec = panel_specs[panel]
    for receptor in spec['receptors']:
        subset = plot_df.loc[
            plot_df['Panel'].eq(panel) & plot_df['Receptor'].eq(receptor)
        ]
        grouped = subset.groupby('Concentration (log M)')['Response']
        means = grouped.mean().sort_index()
        sems = grouped.sem().reindex(means.index)
        ax.errorbar(
            means.index,
            means.values,
            yerr=sems.values,
            fmt=MARKERS[receptor],
            color=COLORS[receptor],
            markersize=6.2,
            markeredgecolor='white',
            markeredgewidth=0.65,
            elinewidth=1.25,
            capsize=2.2,
            capthick=1.1,
            linestyle='none',
            zorder=4,
        )
        fit = fits[(panel, receptor)]
        if fit.get('interpretable', False):
            x_fit = np.linspace(means.index.min(), means.index.max(), 350)
            display_parameters = np.array(fit['parameters'], dtype=float, copy=True)
            display_parameters[1] = min(display_parameters[1], DISPLAY_HILL_SLOPE_CAP)
            ax.plot(
                x_fit,
                model_4pl(x_fit, *display_parameters),
                color=COLORS[receptor],
                linewidth=2.0,
                zorder=3,
            )
            if panel == 'C':
                log_ec50 = float(fit['parameters'][2])
                ax.axvline(
                    log_ec50,
                    color=COLORS[receptor],
                    linewidth=1.1,
                    linestyle='--',
                    alpha=0.85,
                    zorder=2,
                )
                ax.text(
                    log_ec50 + 0.12,
                    0.91,
                    f'{receptor} EC50 = {format_ec50(log_ec50)}',
                    transform=ax.get_xaxis_transform(),
                    color=COLORS[receptor],
                    fontsize=11,
                    fontweight='bold',
                    ha='left',
                    va='top',
                )

    ax.set_title(spec['title'], loc='left', fontweight='bold', pad=8)
    ax.set_xlabel(spec['xlabel'])
    ax.set_ylabel(spec['ylabel'])
    ax.set_xticks(np.arange(-11, -4, 1))
    ax.grid(axis='y', color='#D9D9D9', linewidth=0.7, alpha=0.65)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=4, width=0.9)

    if panel in {'C', 'D'}:
        ax.axhline(0, color='#777777', linewidth=0.9, alpha=0.7, zorder=1)
        if panel == 'D':
            upper_bound = 100
        else:
            # Preserve any observed response that is slightly above the
            # maximum mean carbachol reference rather than clipping it.
            grouped = (
                plot_df.loc[plot_df['Panel'].eq(panel)]
                .groupby(['Receptor', 'Concentration (log M)'], observed=True)['Response']
                .agg(['mean', 'sem'])
            )
            observed_max = np.nanmax((grouped['mean'] + grouped['sem'].fillna(0)).to_numpy())
            upper_bound = max(100, 25 * np.ceil((observed_max + 2) / 25))
        ax.set_ylim(rounded_lower_bound(panel), upper_bound)


fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.2), constrained_layout=False)
for ax, panel in zip(axes.flat, ['A', 'B', 'C', 'D']):
    draw_panel(ax, panel)
    ax.text(
        -0.13, 1.08, panel,
        transform=ax.transAxes,
        fontsize=18,
        fontweight='bold',
        va='top',
        ha='left',
    )

legend_handles = [
    Line2D(
        [0], [0],
        color=COLORS[receptor],
        marker=MARKERS[receptor],
        linestyle='-',
        linewidth=1.8,
        markersize=6.5,
        markeredgecolor='white',
        markeredgewidth=0.6,
        label=receptor,
    )
    for receptor in RECEPTORS
]
fig.legend(
    handles=legend_handles,
    labels=RECEPTORS,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.995),
    ncol=5,
    frameon=False,
    handlelength=2.1,
    columnspacing=1.8,
)
fig.subplots_adjust(left=0.10, right=0.985, bottom=0.085, top=0.92, wspace=0.28, hspace=0.34)

fig.savefig(OUTPUT_PNG, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(OUTPUT_PDF, bbox_inches='tight', facecolor='white')
fig.savefig(OUTPUT_SVG, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Saved PNG: {OUTPUT_PNG}')
print(f'Saved PDF: {OUTPUT_PDF}')
print(f'Saved SVG: {OUTPUT_SVG}')

**Antagonism fixed-carbachol concentrations:** M1, 500 nM; M2, 20 nM; M3, 100 nM; M4, 100 nM; M5, 500 nM.

Panel D is a baseline-centered display: the mean at $-11$ log M is defined as 0% separately for each receptor. This vertical shift does not change the fitted direction, EC50, F-test, or within-series differences.

Significant curve lines use a display-only maximum Hill slope of 2.0. All model-selection tests, potency estimates, and interpretations continue to use the original unconstrained 4PL fits.

Panel C marks the EC50 from the original unconstrained significant fit; the display-only slope cap does not change its horizontal position.

Before this display-only baseline shift, the calcium-flux inhibition calculation uses the explicitly measured carbachol-only control. A same-plate vehicle-only calcium response was not available, so its 100% endpoint denotes complete removal of the carbachol-evoked $\Delta F/F_0$ signal rather than a measured vehicle-control anchor.